In [1]:
include("../LiPoSID.jl")
using QuantumOptics
basis = NLevelBasis(2)
using LinearAlgebra
using HDF5
using DynamicPolynomials
using Plots
using Dates
using Optim
using Random

## Non-Markovian system identification — shared Hamiltonian

We fit the model:
$$\dot\rho(t) = \mathcal{L}(H, J_1)\rho(t) + \mathcal{L}(H, J_m)\rho(t - k\Delta t)$$

Both super-operators share the **same Hamiltonian** $H$ but have independent jump operators $J_1$ and $J_m$.
This gives 3 free variables: $\omega$ (Hamiltonian), $r_1$ (current dissipator), $r_m$ (memory dissipator).

In [ ]:
function scaling_poly(p::Polynomial)
    X = transpose(hcat([exponents(t) for t in terms(p)]...))
    scaling = X \ log.(abs.(coefficients(p)))
    exp.(abs.(scaling))
end

function local_rand_min(p)
    pd = p / maximum(abs.(coefficients(p)))
    scale = scaling_poly(pd)
    p_scaled = subs(pd, variables(pd) => scale .* variables(pd))

    num_iterations = 100
    best_minimizer = nothing
    best_min_value = Inf
    num_of_variables = length(variables(pd))

    for _ in 1:num_iterations
        initial_point = rand(num_of_variables) .* 250
        result = Optim.optimize(p_scaled, initial_point, BFGS())
        if Optim.minimum(result) < best_min_value
            best_minimizer = Optim.minimizer(result)
            best_min_value = Optim.minimum(result)
        end
    end

    best_minimizer = abs.(best_minimizer)
    minimizer_scaled = scale .* best_minimizer
    variables(p_scaled) => minimizer_scaled
end

In [ ]:
"""Simulate non-Markovian dynamics with RK4. Memory uses the simulated history
k steps back; for i ≤ k the initial state ρ[1] is used as constant history."""
function simulate_nonmark_rk4(ρ₀, t, H0, J0, Hm, Jm, k)
    ρ = Vector{Matrix{ComplexF64}}(undef, length(t))
    ρ[1] = ρ₀
    for i in 2:length(t)
        Δt = t[i] - t[i-1]
        i_mem = max(1, i - k)
        ρ_mem = ρ[i_mem]
        Lm = LiPoSID.lindblad_rhs(ρ_mem, Hm, Jm)
        f(ρ_c) = LiPoSID.lindblad_rhs(ρ_c, H0, J0) + Lm
        k1 = f(ρ[i-1])
        k2 = f(ρ[i-1] + Δt/2 * k1)
        k3 = f(ρ[i-1] + Δt/2 * k2)
        k4 = f(ρ[i-1] + Δt * k3)
        ρ[i] = ρ[i-1] + Δt/6 * (k1 + 2k2 + 2k3 + k4)
    end
    return ρ
end

In [ ]:
# Shared Hamiltonian H(w₁); separate jump operators J₁(r₁) and Jm(rm)
@polyvar w₁ r₁ rm

Hˢʸᵐᵇ  = [ w₁   0
             0   0. ]

J₁ˢʸᵐᵇ = [ 0   r₁
            0   0. ]

Hˢʸᵐᵇ  = [ wm   0
             0   0. ]

Jmˢʸᵐᵇ = [ 0   rm
            0   0. ]

In [ ]:
data_dir   = "../DATA/"
models_dir = "../MODELS/"
tests_dir  = "../TESTS/"

dodeca_files = ["State_D"*string(n) for n=1:20]
basis_files  = ["State_B"*string(n) for n=1:4]

train_files = basis_files
test_files  = dodeca_files

k = 5  # memory steps back (τ = k·Δt)

In [ ]:
function g_nonmark_objective(γᵢ)
    obj = 0
    for df_trn in train_files
        ρᵗʳⁿ, tᵗʳⁿ = LiPoSID.get_rho_series(data_dir*df_trn*"_2CUT_data.h5", γᵢ)
        end_train = min(length(tᵗʳⁿ), 1200)
        ρᵗʳⁿ = convert(Vector{Matrix{ComplexF64}}, ρᵗʳⁿ[1:end_train])
        tᵗʳⁿ = convert(Vector{Float64},            tᵗʳⁿ[1:end_train])
        # Same Hamiltonian Hˢʸᵐᵇ for both current-time and memory terms
        obj += LiPoSID.non_mark_obj(ρᵗʳⁿ, tᵗʳⁿ, Hˢʸᵐᵇ, [J₁ˢʸᵐᵇ], Hmˢʸᵐᵇ, [Jmˢʸᵐᵇ], k)
    end
    return obj
end

In [ ]:
γ = ["0.079477", "0.25133", "0.79477", "2.5133", "7.9477", "25.133", "79.477", "251.33"]

date_and_time_string  = string(Dates.format(now(), "yyyy-u-dd_at_HH-MM"))
tests_data_file_name  = "NonMark_sharedH_k$(k)_trn4_tst20_"*date_and_time_string*".h5"

# Safe substitution: convert(Float64, v) is defined for constant Terms (fully substituted),
# but throws for symbolic Terms (absent variables) → default to 0.0.
subs_num(sym, sol) = begin v = subs(sym, sol); try; convert(Float64, v); catch; 0.0; end; end
subs_mat(mat, sol) = map(mat) do e; v = subs(e, sol); try; convert(Float64, v)+0im; catch; 0.0+0im; end; end

"""Project matrix to nearest valid density matrix: Hermitian, PSD, trace-1."""
function to_density_op(basis, ρ)
    ρh = Hermitian((ρ + ρ') / 2)
    F  = eigen(ρh)
    vals = max.(real(F.values), 0.0)
    s = sum(vals)
    s > 0 && (vals ./= s)
    DenseOperator(basis, Matrix(Hermitian(F.vectors * Diagonal(vals) * F.vectors')))
end

for γᵢ in γ
    println("γ = "*γᵢ)

    poly = g_nonmark_objective(γᵢ)
    sol  = local_rand_min(poly)

    omega        = subs_num(w₁,   sol)
    relaxation   = subs_num(r₁^2, sol)
    relaxation_m = subs_num(rm^2, sol)

    Hˢⁱᵈ   = subs_mat(Hˢʸᵐᵇ,  sol)
    J₁ˢⁱᵈ  = subs_mat(J₁ˢʸᵐᵇ, sol)
    J_mˢⁱᵈ = subs_mat(Jmˢʸᵐᵇ, sol)

    println("  ω=$omega,  γ_relax=$relaxation,  γ_m=$relaxation_m")

    h5open(tests_dir*tests_data_file_name, "cw") do fid
        γ_group = create_group(fid, "gamma_"*γᵢ)
        γ_group["omega"]              = omega
        γ_group["gamma_relaxation"]   = relaxation
        γ_group["gamma_relaxation_m"] = relaxation_m
        γ_group["H"]   = Hˢⁱᵈ
        γ_group["k"]   = k
    end

    for df in test_files
        print(df*" ")

        ρₛ, tₛ = LiPoSID.get_rho_series(data_dir*df*"_2CUT_data.h5", γᵢ)
        ρₛ = convert(Vector{Matrix{ComplexF64}}, ρₛ)
        tₛ = convert(Vector{Float64}, tₛ)

        ρˢⁱᵈ = simulate_nonmark_rk4(ρₛ[1], tₛ, Hˢⁱᵈ, [J₁ˢⁱᵈ], Hˢⁱᵈ, [J_mˢⁱᵈ], k)

        ρᵗˢᵗ    = [DenseOperator(basis, Hermitian(ρₜ))  for ρₜ in ρₛ]
        ρˢⁱᵈ_op = [to_density_op(basis, ρₜ)             for ρₜ in ρˢⁱᵈ]

        Fₑₓ = [abs(fidelity(ρ₁, ρ₂)) for (ρ₁, ρ₂) in zip(ρᵗˢᵗ, ρˢⁱᵈ_op)]

        h5open(tests_dir*tests_data_file_name, "cw") do fid
            γ_group          = open_group(fid, "gamma_"*γᵢ)
            init_state_group = create_group(γ_group, df)
            init_state_group["Fidelity"] = convert.(Float64, Fₑₓ)
        end

        println(minimum(Fₑₓ))
    end

    println()
end

println(tests_data_file_name)